# MOD-07 · NB-03 — Spatial Regression: Lag, Error & GWR
### Health Analytics with Python · Module 07: Geospatial & Spatial Epidemiology
---
**Learning objectives**
- Diagnose spatial autocorrelation in OLS residuals with Lagrange Multiplier tests
- Fit Spatial Lag Model (SLM): includes spatially lagged outcome
- Fit Spatial Error Model (SEM): includes spatially autocorrelated errors
- Compare model fit with AIC, log-likelihood, and pseudo-R²
- Implement Geographically Weighted Regression (GWR) from scratch
- Map spatially varying coefficients from GWR

**Estimated time:** 3 hours | **Level:** Advanced  
**Libraries:** `numpy`, `scipy`, `sklearn`, `matplotlib`


## 1. Setup & Dataset

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
from scipy.ndimage import gaussian_filter; from scipy.spatial import cKDTree
warnings.filterwarnings("ignore"); import os; os.makedirs("/tmp/mod07", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})
np.random.seed(42)
NROW,NCOL=10,20; N=NROW*NCOL
row_idx=np.repeat(np.arange(NROW),NCOL); col_idx=np.tile(np.arange(NCOL),NROW)
lon=-120+col_idx*2.5+np.random.normal(0,0.3,N)
lat=28+row_idx*2.0+np.random.normal(0,0.2,N)
pct_poverty=10+8*np.random.beta(2,5,N)+3*np.sin(lon/20)
pct_uninsured=8+6*np.random.beta(2,4,N)+0.3*pct_poverty
pm25=8+4*np.random.gamma(2,1,N)
population=np.random.lognormal(10.5,1.0,N).astype(int)
def sp(vals,sigma=2):
    grid=np.zeros((NROW,NCOL))
    for r,c,v in zip(row_idx,col_idx,vals): grid[r,c]=v
    return gaussian_filter(grid,sigma=sigma)[row_idx,col_idx]
spatial_risk=sp(np.random.normal(0,1,N))
cvd_rate=(180+1.2*pct_poverty+0.8*pct_uninsured+0.5*pm25
          +15*spatial_risk+np.random.normal(0,12,N))
expected=(cvd_rate/100000)*population*5
observed=np.random.poisson(expected).astype(int)
smr=observed/expected.clip(1)
df=pd.DataFrame({"county_id":[f"C{i:03d}" for i in range(N)],
    "lon":lon,"lat":lat,"row":row_idx,"col":col_idx,
    "pct_poverty":pct_poverty,"pct_uninsured":pct_uninsured,"pm25":pm25,
    "population":population,"cvd_rate":cvd_rate,
    "observed":observed,"expected":expected,"smr":smr,"spatial_risk":spatial_risk})
# Build queen weights
def build_queen(row_idx,col_idx):
    N=len(row_idx); W=np.zeros((N,N))
    for i in range(N):
        for j in range(N):
            if i!=j:
                if max(abs(row_idx[i]-row_idx[j]),abs(col_idx[i]-col_idx[j]))<=1:
                    W[i,j]=1
    rs=W.sum(axis=1,keepdims=True); rs[rs==0]=1; return W/rs
W=build_queen(df.row.values,df.col.values)
COVARIATES=["pct_poverty","pct_uninsured","pm25"]
print(f"N={N} | CVD={cvd_rate.mean():.1f}/100k | Weights built")

## 2. OLS Baseline and Diagnostic Tests

**OLS model:** CVD_i = β₀ + β₁·Poverty_i + β₂·Uninsured_i + β₃·PM25_i + ε_i

**Spatial diagnostics:**
- **Moran's I on residuals**: global test for residual autocorrelation
- **Lagrange Multiplier (LM) Lag test**: tests if spatial lag of Y belongs in model
- **LM Error test**: tests if spatial error structure is needed
- **Robust LM tests**: account for presence of the other spatial effect


In [ ]:
import statsmodels.api as sm
from scipy import stats

# OLS
X = sm.add_constant(df[COVARIATES])
y = df["cvd_rate"]
ols = sm.OLS(y, X).fit()
resid = ols.resid.values
print("OLS Results:")
print(ols.summary().tables[1])
print(f"\nAIC: {ols.aic:.2f} | Log-L: {ols.llf:.2f} | R²: {ols.rsquared:.4f}")


In [ ]:
# Moran's I on residuals
def morans_i_resid(resid, W, n_perm=499, seed=42):
    n=len(resid); z=resid-resid.mean(); S0=W.sum()
    I_obs=(n/S0)*(z@(W@z))/(z@z)
    rng=np.random.default_rng(seed)
    I_perm=np.array([(n/S0)*(rng.permutation(z)@(W@rng.permutation(z)))/(z@z) for _ in range(n_perm)])
    p=(np.abs(I_perm)>=np.abs(I_obs)).mean()
    return I_obs, p

I_ols, p_ols = morans_i_resid(resid, W)
print(f"Moran's I (OLS residuals): {I_ols:.4f}, p={p_ols:.4f}")
if p_ols < 0.05:
    print("-> Significant residual spatial autocorrelation: spatial model needed")

# LM tests (simplified)
Wy = W @ y.values
e = resid
We = W @ e
n = len(y)
S1 = 0.5 * np.sum((W + W.T)**2)
S2 = np.sum((W.sum(axis=1) + W.T.sum(axis=1))**2)
sigma2 = (e@e)/n
J = (1/sigma2) * (Wy @ Wy - 2*(Wy @ e)/(sigma2) * (e @ e))

# LM lag
b1 = X.values.T @ X.values
WX = W @ X.values
T_val = (X.values.T @ Wy) / (X.values.T @ X.values)
LM_lag = (e @ Wy)**2 / (sigma2**2 * (Wy @ Wy - Wy @ X.values @ np.linalg.lstsq(b1, X.values.T @ Wy, rcond=None)[0]))
# LM error (simplified)
LM_err = (e @ We)**2 / (sigma2**2 * np.trace(W.T @ W + W @ W))
LM_lag = max(0, abs(LM_lag)); LM_err = max(0, abs(LM_err))

print(f"LM Lag test: {LM_lag:.4f} (p={1-stats.chi2.cdf(LM_lag,1):.4f})")
print(f"LM Err test: {LM_err:.4f} (p={1-stats.chi2.cdf(LM_err,1):.4f})")
print()
print("Decision rule: If both significant, choose the one with larger test statistic")
if LM_lag > LM_err:
    print("  -> Spatial Lag Model preferred")
else:
    print("  -> Spatial Error Model preferred")


## 3. Spatial Lag Model (SLM)

In [ ]:
# SLM: Y = rho*W*Y + X*beta + e
# Estimated via maximum likelihood (iterated approach)

def spatial_lag_model(y, X, W, rho_grid=None, max_iter=100):
    """
    Fit Spatial Lag Model via concentrated log-likelihood.
    rho is estimated via grid search on log-likelihood.
    """
    y_arr = y.values if hasattr(y,"values") else y
    X_arr = X.values if hasattr(X,"values") else X
    n = len(y_arr)
    if rho_grid is None:
        rho_grid = np.linspace(-0.99, 0.99, 100)

    # For each rho, compute log-likelihood
    # ln L = -n/2*ln(2*pi) - n/2*ln(sigma2) + ln|I-rho*W| - e'e/(2*sigma2)
    from scipy.linalg import det
    I_n = np.eye(n)
    log_liks = []
    for rho in rho_grid:
        A = I_n - rho * W
        Ay = A @ y_arr
        # OLS of Ay on X
        b  = np.linalg.lstsq(X_arr, Ay, rcond=None)[0]
        e  = Ay - X_arr @ b
        s2 = (e@e)/n
        # Log-determinant (use sign + logabsdet for stability)
        sign, logabsdet = np.linalg.slogdet(A)
        if sign <= 0:
            log_liks.append(-1e10)
            continue
        ll = -n/2*np.log(2*np.pi) - n/2*np.log(s2) + logabsdet - n/2
        log_liks.append(ll)
    rho_best = rho_grid[np.argmax(log_liks)]
    # Final fit at best rho
    A  = I_n - rho_best * W
    Ay = A @ y_arr
    b  = np.linalg.lstsq(X_arr, Ay, rcond=None)[0]
    e  = Ay - X_arr @ b
    s2 = (e@e)/n
    ll_best = max(log_liks)
    return {"rho":rho_best, "beta":b, "sigma2":s2, "log_lik":ll_best,
            "residuals":e, "rho_grid":rho_grid, "log_liks":log_liks}

X_arr = sm.add_constant(df[COVARIATES])
slm = spatial_lag_model(y, X_arr, W)
print(f"Spatial Lag Model:")
print(f"  rho (spatial lag)  : {slm['rho']:.4f}")
print(f"  beta (intercept)   : {slm['beta'][0]:.4f}")
feat_names = ["Intercept"]+COVARIATES
for name, b in zip(feat_names, slm["beta"]):
    print(f"  beta ({name:15s}): {b:.4f}")
print(f"  Log-likelihood     : {slm['log_lik']:.4f}")
print(f"  AIC                : {-2*slm['log_lik'] + 2*(len(slm['beta'])+2):.4f}")
print(f"  OLS AIC            : {ols.aic:.4f}")


## 4. Spatial Error Model (SEM)

In [ ]:
def spatial_error_model(y, X, W, lambda_grid=None):
    """
    Fit Spatial Error Model: Y = X*beta + u, u = lambda*W*u + e
    Equivalent to GLS with spatial covariance: (I - lambda*W)
    Estimated via ML grid search on lambda.
    """
    y_arr = y.values if hasattr(y,"values") else y
    X_arr = X.values if hasattr(X,"values") else X
    n = len(y_arr)
    if lambda_grid is None:
        lambda_grid = np.linspace(-0.99, 0.99, 100)
    I_n = np.eye(n)
    log_liks = []
    for lam in lambda_grid:
        B = I_n - lam * W
        By = B @ y_arr
        BX = B @ X_arr
        b  = np.linalg.lstsq(BX, By, rcond=None)[0]
        e  = By - BX @ b
        s2 = (e@e)/n
        sign, logabsdet = np.linalg.slogdet(B)
        if sign <= 0:
            log_liks.append(-1e10); continue
        ll = -n/2*np.log(2*np.pi) - n/2*np.log(s2) + logabsdet - n/2
        log_liks.append(ll)
    lam_best = lambda_grid[np.argmax(log_liks)]
    B = I_n - lam_best*W
    By = B @ y_arr; BX = B @ X_arr
    b  = np.linalg.lstsq(BX, By, rcond=None)[0]
    e  = (y_arr - X_arr @ b)  # original residuals
    s2 = ((B@e)@(B@e))/n
    return {"lambda":lam_best,"beta":b,"sigma2":s2,"log_lik":max(log_liks),"residuals":e}

sem = spatial_error_model(y, X_arr, W)
print(f"Spatial Error Model:")
print(f"  lambda (spatial error): {sem['lambda']:.4f}")
for name, b in zip(feat_names, sem["beta"]):
    print(f"  beta ({name:15s}) : {b:.4f}")
print(f"  Log-likelihood: {sem['log_lik']:.4f}")
print(f"  AIC           : {-2*sem['log_lik'] + 2*(len(sem['beta'])+2):.4f}")
print()
print("Model comparison:")
print(f"  OLS AIC: {ols.aic:.2f} | SLM AIC: {-2*slm['log_lik']+2*(len(slm['beta'])+2):.2f} | SEM AIC: {-2*sem['log_lik']+2*(len(sem['beta'])+2):.2f}")
print(f"  True poverty coefficient: 1.20 | OLS: {ols.params['pct_poverty']:.3f} | SLM: {slm['beta'][1]:.3f} | SEM: {sem['beta'][1]:.3f}")


## 5. Geographically Weighted Regression (GWR)

In [ ]:
# GWR: estimates local coefficients at each location
# Using Gaussian kernel weights: k_ij = exp(-d_ij^2 / (2*h^2))

def gwr(y, X, coords, bandwidth=None):
    """
    Geographically Weighted Regression.
    bandwidth: kernel bandwidth in coordinate units (default: adaptive)
    """
    y_arr = y.values if hasattr(y,"values") else y
    X_arr = X.values if hasattr(X,"values") else X
    n, p  = X_arr.shape
    if bandwidth is None:
        # Adaptive bandwidth: use 30% of observations
        from scipy.spatial.distance import cdist
        dists_all = cdist(coords, coords)
        bandwidth = np.percentile(dists_all, 30)

    from scipy.spatial.distance import cdist
    dists = cdist(coords, coords)
    local_betas = np.zeros((n, p))
    local_pred  = np.zeros(n)
    local_resid = np.zeros(n)

    for i in range(n):
        # Gaussian kernel weights
        w_i = np.exp(-dists[i]**2 / (2*bandwidth**2))
        W_i = np.diag(w_i)
        # Weighted OLS
        XtW  = X_arr.T @ W_i
        XtWX = XtW @ X_arr
        XtWy = XtW @ y_arr
        try:
            b_i = np.linalg.solve(XtWX, XtWy)
        except np.linalg.LinAlgError:
            b_i = np.linalg.lstsq(XtWX, XtWy, rcond=None)[0]
        local_betas[i]  = b_i
        local_pred[i]   = X_arr[i] @ b_i
        local_resid[i]  = y_arr[i] - local_pred[i]
    return {"betas":local_betas,"pred":local_pred,"resid":local_resid,"bandwidth":bandwidth}

print("Fitting GWR (this may take 30-60 seconds)...")
coords_arr = np.column_stack([df.lon, df.lat])
gwr_res = gwr(y, X_arr, coords_arr, bandwidth=8.0)
print(f"GWR bandwidth: {gwr_res['bandwidth']:.1f} degrees")
df_gwr = X_arr.copy()
for i, name in enumerate(feat_names):
    df[f"gwr_beta_{name.replace(' ','_')}"] = gwr_res["betas"][:,i]
df["gwr_pred"]  = gwr_res["pred"]
df["gwr_resid"] = gwr_res["resid"]
sse_gwr = (gwr_res["resid"]**2).sum()
sse_ols = (ols.resid**2).sum()
print(f"SSE improvement: OLS={sse_ols:.0f} -> GWR={sse_gwr:.0f} ({(1-sse_gwr/sse_ols)*100:.1f}% reduction)")


In [ ]:
# Map spatially varying coefficients from GWR
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
covariate_plot = ["pct_poverty", "pct_uninsured", "pm25"]
beta_true = [1.2, 0.8, 0.5]
cmap_gwr = "RdBu_r"

for ax, cov, true_b in zip(axes, covariate_plot, beta_true):
    beta_col = f"gwr_beta_{cov}"
    vals = df[beta_col]
    vmin, vmax = vals.quantile(0.05), vals.quantile(0.95)
    sc = ax.scatter(df.lon, df.lat, c=vals, cmap=cmap_gwr,
                    vmin=vmin, vmax=vmax, s=150, alpha=0.85, edgecolors="white", linewidth=0.2)
    plt.colorbar(sc, ax=ax, label="Local coefficient")
    ax.set_title(f"GWR: Local β for {cov}
Global OLS β={ols.params[cov]:.3f} | True β={true_b}")
    ax.set_xlabel("Lon"); ax.set_ylabel("Lat")
    ax.axhline(df.lat.mean(), color="gray", ls="--", lw=0.8, alpha=0.5)

plt.suptitle("GWR Spatially Varying Coefficients", fontsize=12)
plt.tight_layout()
plt.savefig("/tmp/mod07/gwr_coefficients.png", bbox_inches="tight"); plt.show()
print("GWR coefficient statistics:")
for cov in covariate_plot:
    b_col = df[f"gwr_beta_{cov}"]
    print(f"  {cov:20s}: mean={b_col.mean():.3f} | SD={b_col.std():.3f} | range=[{b_col.min():.2f},{b_col.max():.2f}]")


## 6. Knowledge Check
1. Spatial lag model rho = 0.38. Interpret this parameter in clinical terms.
2. SEM has lower AIC than SLM. Under what conceptual assumption should you prefer SEM?
3. GWR shows poverty coefficient varying from 0.4 to 2.8 across counties. Is this capturing true heterogeneity or spatial artefact?
4. OLS coefficient for poverty = 1.42, SLM = 0.95. Why does controlling for spatial autocorrelation reduce the poverty coefficient?
5. Run GWR with two different bandwidths (h=5 and h=15). How does bandwidth affect the spatial smoothness of local coefficients?


In [ ]:
# Exercise 5 — GWR bandwidth sensitivity
print("GWR bandwidth sensitivity (small sample, fast demo):")
for bw in [4.0, 8.0, 15.0]:
    gwr_bw = gwr(y.iloc[:50], X_arr.iloc[:50], coords_arr[:50], bandwidth=bw)
    beta_pov = gwr_bw["betas"][:,1]  # poverty coefficient
    sse_bw = (gwr_bw["resid"]**2).sum()
    print(f"  bandwidth={bw:.0f}: poverty β range=[{beta_pov.min():.2f},{beta_pov.max():.2f}] SD={beta_pov.std():.3f} SSE={sse_bw:.1f}")
print()
print("Smaller bandwidth -> more local variation, higher risk of overfitting")
print("Larger bandwidth  -> smoother coefficients, approaches global OLS")


---
**Next → NB-04: Disease Mapping & Bayesian Smoothing**